# 01 · Bronze: ingesta de los CSV de Olist

**Objetivo:** copiar los 9 CSV de `Files/olist/` a tablas Delta en el esquema `bronze`, **sin interpretar nada**.

**Reglas de Bronze:**
1. Todas las columnas entran como **texto** (`inferSchema = False`): la limpieza y los tipos se hacen en Silver.
2. Se añaden dos columnas de trazabilidad: `_ingested_at` (cuándo) y `_source_file` (de qué archivo).
3. Carga **idempotente**: `mode("overwrite")`, así que ejecutarlo dos veces da el mismo resultado.
4. `multiLine` y `escape` permiten leer las reseñas, cuyos comentarios tienen saltos de línea y comillas.

In [ ]:
from pyspark.sql import functions as F

archivos = {
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

In [ ]:
for tabla, archivo in archivos.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .option("inferSchema", False)
          .csv(f"Files/olist/{archivo}")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(archivo)))
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta")
       .saveAsTable(f"bronze.{tabla}"))
    print(f"bronze.{tabla}: {spark.table(f'bronze.{tabla}').count()} filas")

## Verificación
Filas esperadas: orders 99.441 · order_items 112.650 · order_payments 103.886 · order_reviews 99.224 · customers 99.441 · products 32.951 · sellers 3.095 · geolocation 1.000.163 · category_translation 71.

In [ ]:
esperado = {"orders": 99441, "order_items": 112650, "order_payments": 103886, "order_reviews": 99224,
            "customers": 99441, "products": 32951, "sellers": 3095, "geolocation": 1000163,
            "category_translation": 71}
errores = []
for tabla, n in esperado.items():
    real = spark.table(f"bronze.{tabla}").count()
    estado = "OK" if real == n else "ERROR"
    print(f"{estado:6} bronze.{tabla}: esperado {n} / real {real}")
    if real != n:
        errores.append(tabla)
# Control: si algún conteo no coincide, el notebook falla y el pipeline se detiene.
assert not errores, f"Conteos distintos en: {errores}"